In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [7]:
df = pd.read_csv("../data/cleaned_customer_data.csv")
df.head()

,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_cancellation,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0


In [9]:
# Create a connection to an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Write a cleaned pandas DataFrame to a table named 'customer_sales'
df.to_sql("customer_sales", conn, index=False, if_exists="replace")

407664

Calculate customer KPIs

In [15]:
total_customer = """SELECT COUNT(DISTINCT customer_id) AS total_customer FROM customer_sales"""
total_customer_result = pd.read_sql(total_customer, conn)
display(total_customer_result)

,total_customer
0,4312


In [17]:
total_order = """SELECT COUNT(DISTINCT invoice) AS total_orders FROM customer_sales"""
total_order_result = pd.read_sql(total_order, conn)
display(total_order_result)

,total_orders
0,19213


In [18]:
total_revenue = """SELECT SUM(revenue) AS total_revenue FROM customer_sales"""
total_revenue_result = pd.read_sql(total_revenue, conn)
display(total_revenue_result)

,total_revenue
0,8.832003e+06


In [22]:
avg_customer_spending = """SELECT AVG(customer_revenue) AS avg_customer_spending
FROM(SELECT customer_id, SUM(revenue) AS customer_revenue FROM customer_sales GROUP BY customer_id) AS customers"""
avg_customer_spending_result = pd.read_sql(avg_customer_spending, conn)
display(avg_customer_spending_result)

,avg_customer_spending
0,2048.238236


Calculate the number of orders per customer

In [23]:
orders_per_customer = """SELECT customer_id, COUNT(DISTINCT invoice) AS total_orders FROM customer_sales 
GROUP BY customer_id ORDER BY total_orders DESC"""
order_per_customer_result = pd.read_sql(orders_per_customer, conn)
display(order_per_customer_result)

,customer_id,total_orders
0,14911.0,205
1,17850.0,155
2,12748.0,144
3,15311.0,121
4,13089.0,109
...,...,...
4307,12362.0,1
4308,12355.0,1
4309,12353.0,1
4310,12351.0,1


Identify repeat customers

In [27]:
repeated_customers = """SELECT customer_id, COUNT(DISTINCT invoice) as total_orders FROM customer_sales
GROUP BY customer_id Having total_orders > 1"""
repeated_customers_result = pd.read_sql(repeated_customers, conn)
display(repeated_customers_result)

,customer_id,total_orders
0,12346.0,11
1,12347.0,2
2,12349.0,3
3,12352.0,2
4,12356.0,3
...,...,...
2888,18276.0,5
2889,18277.0,4
2890,18283.0,6
2891,18286.0,2


Calculate repeat purchase rate

In [31]:
repeat_purchase_rate = """WITH customer_orders AS (SELECT customer_id, COUNT(DISTINCT invoice) AS total_orders 
FROM customer_sales GROUP BY customer_id)
SELECT COUNT(*) AS total_customers, COUNT(*) FILTER (WHERE total_orders > 1) AS repeat_customers,
ROUND(100.0 * COUNT(*) FILTER (WHERE total_orders > 1) / COUNT(*), 2) AS repeat_purchase_rate FROM customer_orders;"""
repeat_purchase_rate_result = pd.read_sql(repeat_purchase_rate, conn)
display(repeat_purchase_rate_result)

,total_customers,repeat_customers,repeat_purchase_rate
0,4312,2893,67.09


Identify the top 10 customers

In [33]:
top_10_customers = """SELECT customer_id, SUM(revenue) AS total_spending, COUNT(DISTINCT invoice) AS total_orders 
FROM customer_sales GROUP BY customer_id ORDER BY total_spending DESC LIMIT 10"""
top_10_customers_result = pd.read_sql(top_10_customers, conn)
display(top_10_customers_result)

,customer_id,total_spending,total_orders
0,18102.0,349164.35,89
1,14646.0,248396.50,78
2,14156.0,196566.74,102
3,14911.0,152147.57,205
4,13694.0,131443.19,94
5,17511.0,84541.17,31
6,15061.0,83284.38,86
7,16684.0,80489.21,27
8,16754.0,65500.07,29
9,17949.0,60117.60,74


Analyze customer purchasing frequency

In [34]:
purchasing_frequency = """SELECT customer_id, COUNT(DISTINCT invoice) AS total_orders, SUM(revenue) AS total_spending, 
AVG(revenue) AS avg_line_revenue FROM customer_sales GROUP by customer_id ORDER BY total_orders DESC"""
purchasing_frequency_result = pd.read_sql(purchasing_frequency, conn)
display(purchasing_frequency_result)

,customer_id,total_orders,total_spending,avg_line_revenue
0,14911.0,205,152147.57,27.315542
1,17850.0,155,51208.87,20.491745
2,12748.0,144,22879.66,8.689578
3,15311.0,121,56003.26,27.252195
4,13089.0,109,57912.03,37.507791
...,...,...,...,...
4307,12362.0,1,130.00,130.000000
4308,12355.0,1,488.21,22.191364
4309,12353.0,1,317.76,15.888000
4310,12351.0,1,300.93,14.330000


In [36]:
df.to_csv("../data/01_customer_kpis.csv", index=False)